# Data Pipeline

## Problem Statement

Zepto's analysts need a way to benchmark catalog-style pricing and availability data before it ever reaches a dashboard.

**Objective:**
In this module, we scrape live product data from a public scraping-practice site, clean it, enrich it with the project's baseline fixed-rate currency conversion, and load it into a properly normalized relational database that we then query with both SQL and pandas — exactly the kind of raw-to-relational pipeline a catalog/competitive-intelligence workflow needs.


## Web Scraping

In this task, we scrape book data from the **Books to Scrape** website using **Requests** and **BeautifulSoup**. The collected raw dataset will be used in the subsequent stages of the pipeline for cleaning, transformation, database loading, and analysis.


In [1]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin


In [2]:
import time
HEADER = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/58.0.3029.110 Safari/537.3"
}
def get_page(url, retries=3, delay=1):
    for attempt in range(retries):
        try:
            response = requests.get(url, headers=HEADER)
            response.raise_for_status()
            return response
        except requests.exceptions.RequestException as e:
            print(f"Request failed for {url}: {e}, Retrying ({attempt + 1}/{retries})... ")
            time.sleep(delay)
    
    print(f"Failed to retrieve page for {url} after {retries} attempts.")
    return None


In [3]:
BASE_URL = "https://books.toscrape.com/"
response = get_page(BASE_URL)
print(f"Status Code: {response.status_code}")


Status Code: 200


In [4]:
soup = BeautifulSoup(response.content, 'html.parser')


### Extract Book Categories


In [5]:
sidebar_categories = soup.find('div', class_='side_categories')
get_categories = sidebar_categories.find_all('a')

categories = []

for category in get_categories[1:]:
    category_name = category.text.strip()
    category_url = urljoin(BASE_URL, category['href'])
    categories.append({
        'name': category_name,
        'url': category_url
    })

print(f"Total Categories: {len(categories)}")


Total Categories: 50


In [6]:
print("Categories with urls:\n")
for i, category in enumerate(categories):
    print(f"Category {i+1}: {category['name']} - {category['url']}")


Categories with urls:

Category 1: Travel - https://books.toscrape.com/catalogue/category/books/travel_2/index.html
Category 2: Mystery - https://books.toscrape.com/catalogue/category/books/mystery_3/index.html
Category 3: Historical Fiction - https://books.toscrape.com/catalogue/category/books/historical-fiction_4/index.html
Category 4: Sequential Art - https://books.toscrape.com/catalogue/category/books/sequential-art_5/index.html
Category 5: Classics - https://books.toscrape.com/catalogue/category/books/classics_6/index.html
Category 6: Philosophy - https://books.toscrape.com/catalogue/category/books/philosophy_7/index.html
Category 7: Romance - https://books.toscrape.com/catalogue/category/books/romance_8/index.html
Category 8: Womens Fiction - https://books.toscrape.com/catalogue/category/books/womens-fiction_9/index.html
Category 9: Fiction - https://books.toscrape.com/catalogue/category/books/fiction_10/index.html
Category 10: Childrens - https://books.toscrape.com/catalogue/cat

### Collect Book Links


In [7]:
def get_category_book_urls(category_url):
    book_urls = []
    current_page = category_url
    while current_page:
        response = get_page(current_page)

        soup = BeautifulSoup(response.content, 'html.parser')
        books = soup.find_all('article', class_='product_pod')

        for book in books:
            book_url = book.find('h3').find('a')['href']

            absolute_url = urljoin(current_page, book_url)
            book_urls.append(absolute_url)

        next_button = soup.find('li', class_='next')

        if next_button:
            next_page_url = next_button.find('a')['href']
            current_page = urljoin(current_page, next_page_url)
        else:
            current_page = None

    return book_urls
         

In [8]:
print("Number of Books in Each Category:\n")
for i, category in enumerate(categories):
    print(f"{i+1} Category: {category['name']} --- No of books: {len(get_category_book_urls(category['url']))}")


Number of Books in Each Category:

1 Category: Travel --- No of books: 11
2 Category: Mystery --- No of books: 32
3 Category: Historical Fiction --- No of books: 26
4 Category: Sequential Art --- No of books: 75
5 Category: Classics --- No of books: 19
6 Category: Philosophy --- No of books: 11
7 Category: Romance --- No of books: 35
8 Category: Womens Fiction --- No of books: 17
9 Category: Fiction --- No of books: 65
10 Category: Childrens --- No of books: 29
11 Category: Religion --- No of books: 7
12 Category: Nonfiction --- No of books: 110
13 Category: Music --- No of books: 13
14 Category: Default --- No of books: 152
15 Category: Science Fiction --- No of books: 16
16 Category: Sports and Games --- No of books: 5
17 Category: Add a comment --- No of books: 67
18 Category: Fantasy --- No of books: 48
19 Category: New Adult --- No of books: 6
20 Category: Young Adult --- No of books: 54
21 Category: Science --- No of books: 14
22 Category: Poetry --- No of books: 19
23 Category: 

### Extract Book Details


In [9]:
def get_book_details(book_url):
    response = get_page(book_url)

    soup = BeautifulSoup(response.content, 'html.parser')

    title = soup.find('div', class_='product_main').find('h1').text.strip()

    price = soup.find('p', class_='price_color').text.strip()

    star_rating = soup.find('p', class_='star-rating')['class'][1]

    availability = soup.find('p', class_='instock availability').text.strip()

    breadcrumb = soup.find('ul', class_='breadcrumb').find_all('li')
    category = breadcrumb[2].text.strip()

    return {
        'title': title,
        'price': price,
        'star_rating': star_rating,
        'availability': availability,
        'category': category
    }
    

### Scrape Selected Categories


In [10]:
selected_categories = ['Sequential Art', 'Fiction', 'Nonfiction', 'Young Adult']
all_books = []


In [11]:
category_url_map = {c['name']: c['url'] for c in categories}

for selected_category in selected_categories:
    category_url = category_url_map.get(selected_category)
    book_urls = get_category_book_urls(category_url)
    for book_url in book_urls:
        book = get_book_details(book_url)
        all_books.append(book)
    print(f"Retrieved books for category: {selected_category}")
print(f"Total number of books collected: {len(all_books)}")


Retrieved books for category: Sequential Art
Retrieved books for category: Fiction
Retrieved books for category: Nonfiction
Retrieved books for category: Young Adult
Total number of books collected: 304


## Data Cleaning

Clean the scraped book data by converting fields into appropriate data types and handling parsing failures without interrupting the pipeline.


In [12]:
import pandas as pd

books_df = pd.DataFrame(all_books)
print(f"DataFrame Shape: {books_df.shape}")

books_df_clean = books_df.copy()


DataFrame Shape: (304, 5)


In [13]:
books_df_clean.index = books_df.index + 1
books_df_clean.head()


,title,price,star_rating,availability,category
1,Scott Pilgrim's Precious Little Life (Scott Pi...,£52.29,Five,In stock (19 available),Sequential Art
2,Tsubasa: WoRLD CHRoNiCLE 2 (Tsubasa WoRLD CHRo...,£16.28,One,In stock (16 available),Sequential Art
3,This One Summer,£19.49,Four,In stock (16 available),Sequential Art
4,The Nameless City (The Nameless City #1),£38.16,Four,In stock (16 available),Sequential Art
5,"Saga, Volume 5 (Saga (Collected Editions) #5)",£51.04,Two,In stock (16 available),Sequential Art


### Strip Currency Symbol


In [14]:
books_df_clean['price_gbp'] = pd.to_numeric(books_df_clean['price'].str.replace('£', '', regex=False).astype(float), errors='coerce')
books_df_clean.drop(columns=['price'], inplace=True)


### Convert Star Ratings


In [15]:
RATING = {
    "one": 1,
    "two": 2,
    "three": 3,
    "four": 4,
    "five": 5
}

books_df_clean['rating'] = books_df_clean['star_rating'].str.lower().map(RATING)
books_df_clean.drop(columns=['star_rating'], inplace=True)


### Parse availability to in_stock


In [16]:
books_df_clean['in_stock'] = books_df_clean['availability'].str.lower().str.contains('in stock', na=False).astype(int)
books_df_clean.drop(columns=['availability'], inplace=True)


### Handling Parsing Failures

To ensure the pipline remains robust:

- Numeric fileds `price_gbp` and `rating` use **Median Imputation** when parsing fails. The median is less affected by outliers than the mean and preserves the central tendency of the data.
- Rows with missing essential text fields `title` or `category` are **dropped**, since these values cannot be reliably inferred and are required for meaningful analysis.


In [17]:
books_df_clean['price_gbp'] = books_df_clean['price_gbp'].fillna(books_df_clean['price_gbp'].median())
books_df_clean['rating'] = books_df_clean['rating'].fillna(books_df_clean['rating'].median())


In [18]:
books_df_clean.head()


,title,category,price_gbp,rating,in_stock
1,Scott Pilgrim's Precious Little Life (Scott Pi...,Sequential Art,52.29,5,1
2,Tsubasa: WoRLD CHRoNiCLE 2 (Tsubasa WoRLD CHRo...,Sequential Art,16.28,1,1
3,This One Summer,Sequential Art,19.49,4,1
4,The Nameless City (The Nameless City #1),Sequential Art,38.16,4,1
5,"Saga, Volume 5 (Saga (Collected Editions) #5)",Sequential Art,51.04,2,1


In [19]:
print(books_df_clean.info())


<class 'pandas.DataFrame'>
RangeIndex: 304 entries, 1 to 304
Data columns (total 5 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   title      304 non-null    str    
 1   category   304 non-null    str    
 2   price_gbp  304 non-null    float64
 3   rating     304 non-null    int64  
 4   in_stock   304 non-null    int64  
dtypes: float64(1), int64(2), str(2)
memory usage: 12.0 KB
None


## Currency Conversion

Convert the cleaned book prices from GBP to INR using the project-defined fixed conversion rate.

**Conversion Rate:**

> **1 GBP = 105.50 INR**


In [20]:
GBP_TO_INR_RATE = 105.50


In [21]:
books_df_clean['price_inr'] = (books_df_clean['price_gbp'] * GBP_TO_INR_RATE).round(2)


In [22]:
books_df_clean.head()


,title,category,price_gbp,rating,in_stock,price_inr
1,Scott Pilgrim's Precious Little Life (Scott Pi...,Sequential Art,52.29,5,1,5516.60
2,Tsubasa: WoRLD CHRoNiCLE 2 (Tsubasa WoRLD CHRo...,Sequential Art,16.28,1,1,1717.54
3,This One Summer,Sequential Art,19.49,4,1,2056.19
4,The Nameless City (The Nameless City #1),Sequential Art,38.16,4,1,4025.88
5,"Saga, Volume 5 (Saga (Collected Editions) #5)",Sequential Art,51.04,2,1,5384.72


## Database Schema Design

Create a normalized SQLite database consisting of Categories and Books tables.


In [23]:
import sqlite3

conn = sqlite3.connect('books_catalog.db')
cursor = conn.cursor()


### Create Categories Table


In [24]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS categories (
    category_id INTEGER PRIMARY KEY AUTOINCREMENT,
    category_name TEXT UNIQUE NOT NULL
);
""")


### Create Books Table


In [25]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS books(
    book_id INTEGER PRIMARY KEY AUTOINCREMENT,
    title TEXT NOT NULL,
    price_gbp REAL,
    price_inr REAL,
    rating INTEGER,
    in_stock INTEGER,
    category_id INTEGER,
    FOREIGN KEY (category_id) REFERENCES categories(category_id)
);
""")

conn.commit()


In [26]:
cursor.execute("""
SELECT name FROM sqlite_master WHERE type='table';
""")

tables = cursor.fetchall()

print("Database Tables:")

for table in tables:
    print(table[0])


Database Tables:
categories
sqlite_sequence
books


## Load Data into SQLite Database

Insert the cleaned dataset into the normalized SQLite database and execute SQL queries to validate and analyze the stored data.


In [27]:
categories = books_df_clean["category"].drop_duplicates().sort_values().tolist()
print(categories)


['Fiction', 'Nonfiction', 'Sequential Art', 'Young Adult']


### Insert Categories


In [28]:
cursor.executemany("""
INSERT INTO CATEGORIES (category_name)
VALUES (?);
""", [(category, ) for category in categories])

conn.commit()


In [29]:
cursor.execute("""
SELECT category_id, category_name 
FROM categories;
""")

category_map = {
    category_name: category_id
    for category_id, category_name in cursor.fetchall()
}


### Insert Books


In [30]:
book_records = []
for _, row in books_df_clean.iterrows():
    book_records.append((
        row['title'],
        row['price_gbp'],
        row['price_inr'],
        row['rating'],
        int(row['in_stock']),
        category_map[row['category']]
    ))
print(f"Total number of book records created: {len(book_records)}")


Total number of book records created: 304


In [31]:
cursor.executemany("""
INSERT INTO books (title, price_gbp, price_inr, rating, in_stock, category_id)
VALUES (?, ?, ?, ?, ?, ?);
""", book_records)

conn.commit()


In [32]:
cursor.execute("""
SELECT COUNT(*) FROM categories;
""")

total_categories = cursor.fetchall()[0][0]
print(f"Total categories: {total_categories}")


Total categories: 4


In [33]:
cursor.execute("""
SELECT COUNT (*) FROM books;
""")

total_books = cursor.fetchone()[0]
print(f"Total number of books in the database: {total_books}")


Total number of books in the database: 304


### Query 1: SELECT with WHERE


In [34]:
query_1 = """
SELECT title, rating, price_inr FROM books WHERE rating >= 4;
"""

cursor.execute(query_1)
query_1_output = cursor.fetchall()

print(f"Query 1 Retrieved Rows: {len(query_1_output)}")


Query 1 Retrieved Rows: 124


### Query 2: ORDER BY


In [35]:
query_2 = """
SELECT title, rating, price_inr FROM books ORDER BY price_inr DESC LIMIT 20;
"""

cursor.execute(query_2)
query_2_output = cursor.fetchall()

print(f"Query 2 Retrieved Rows: {len(query_2_output)}")


Query 2 Retrieved Rows: 20


### Query 3: LIMIT


In [36]:
query_3 = """
SELECT title, price_inr FROM books WHERE price_inr > 2000 ORDER BY price_inr DESC LIMIT 20;
"""

cursor.execute(query_3)
query_3_output = cursor.fetchall()

print(f"Query 3 Retrieved Rows: {len(query_3_output)}")


Query 3 Retrieved Rows: 20


### Query 4: DISTINCT


In [37]:
query_4 = """
SELECT DISTINCT category_name FROM categories ORDER BY category_name;
"""

cursor.execute(query_4)
query_4_output = cursor.fetchall()

print(f"Query 4 Retrieved Rows: {len(query_4_output)}")


Query 4 Retrieved Rows: 4


### Query 5: IN


In [38]:
query_5 = """
SELECT title, rating, price_inr
FROM books WHERE rating IN(1, 3, 5)
"""

cursor.execute(query_5)
query_5_output = cursor.fetchall()

print(f"Query 5 Retrieved Rows: {len(query_5_output)}")


Query 5 Retrieved Rows: 191


### Query 6: BETWEEN


In [39]:
query_6 = """
SELECT title, rating, price_inr
FROM books WHERE price_inr BETWEEN 1000 AND 2000;
"""

cursor.execute(query_6)
query_6_output = cursor.fetchall()

print(f"Query 6 Retrieved Rows: {len(query_6_output)}")


Query 6 Retrieved Rows: 53


### Query 7: JOIN


In [40]:
query_7 = """
SELECT b.title, c.category_name, b.rating, b.price_inr
FROM books AS b
INNER JOIN categories AS c ON b.category_id = c.category_id
ORDER BY c.category_name ASC, b.rating DESC, b.title ASC;
"""

cursor.execute(query_7)
query_7_output = cursor.fetchall()

print(f"Query 7 Retrieved Rows: {len(query_7_output)}")


Query 7 Retrieved Rows: 304


## Reproduce & Validate SQL using Pandas

Read SQL query results into pandas DataFrames using `pd.read_sql()` and reproduce the SQL JOIN using `pd.merge()` to verify that both approaches produce equivalent results.


In [41]:
query_2_df = pd.read_sql(query_2, conn)
query_5_df = pd.read_sql(query_5, conn)


In [42]:
query_7_sql_df = pd.read_sql(query_7, conn)


In [43]:
query_7_sql_df = (
    query_7_sql_df
    .sort_values(
        by=["category_name", "rating", "title"],
        ascending=[True, False, True]
    )
    .reset_index(drop=True)
)


In [44]:
books_sql = pd.read_sql("""
SELECT * FROM books;
""", conn)

categories_sql = pd.read_sql("""
SELECT * FROM categories;
""", conn)


In [45]:
merge_df = (
    pd.merge(
        books_sql,
        categories_sql,
        on="category_id",
        how="inner"
    )
    [["title", "category_name", "rating", "price_inr"]]
    .sort_values(
        by=["category_name", "rating", "title"],
        ascending=[True, False, True]
    )
    .reset_index(drop=True)
)


In [46]:
print("Do pd.read_sql() and pd.merge() produce equivalent results?")
print(f"Equivalent Output: {query_7_sql_df.equals(merge_df)}\n\n")

print("SQL JOIN Result (pd.read_sql)")
display(query_7_sql_df)

print("Pandas Merge Result (pd.merge)")
display(merge_df)


Do pd.read_sql() and pd.merge() produce equivalent results?
Equivalent Output: True


SQL JOIN Result (pd.read_sql)


,title,category_name,rating,price_inr
0,Atlas Shrugged,Fiction,5,2804.19
1,Bright Lines,Fiction,5,4121.88
2,Crazy Rich Asians (Crazy Rich Asians #1),Fiction,5,5183.22
3,Dear Mr. Knightley,Fiction,5,1182.66
4,Digital Fortress,Fiction,5,6119.00
...,...,...,...,...
299,South of Sunshine,Young Adult,1,3052.12
300,Tell Me Three Things,Young Adult,1,4410.96
301,The Alien Club,Young Adult,1,5739.20
302,The Requiem Red,Young Adult,1,2389.57


Pandas Merge Result (pd.merge)


,title,category_name,rating,price_inr
0,Atlas Shrugged,Fiction,5,2804.19
1,Bright Lines,Fiction,5,4121.88
2,Crazy Rich Asians (Crazy Rich Asians #1),Fiction,5,5183.22
3,Dear Mr. Knightley,Fiction,5,1182.66
4,Digital Fortress,Fiction,5,6119.00
...,...,...,...,...
299,South of Sunshine,Young Adult,1,3052.12
300,Tell Me Three Things,Young Adult,1,4410.96
301,The Alien Club,Young Adult,1,5739.20
302,The Requiem Red,Young Adult,1,2389.57


In [47]:
conn.close()
print("SQLite connection closed.")


SQLite connection closed.


# Conclusion

This notebook successfully implements the pipeline by:

- Scraping book catalog data
- Cleaning and validating the dataset
- Converting prices from GBP to INR
- Creating a normalized SQLite database
- Executing SQL queries
- Validating SQL JOIN results using pandas
